# Chatbot and RAG Evaluation

Retrieval-Augmented Generation (RAG) improves a Large Language Model (LLM) by supplying relevant external context before generation. This helps the model produce answers that are more grounded, accurate, and useful for domain-specific chatbot applications.

In this notebook, we will build and evaluate a chatbot pipeline using `LangSmith` for tracing and evaluation, `Groq` as the LLM provider, and `llama-3.1-8b-instant` as the model used for response generation.

## What We Will Do

1. Create evaluation datasets with sample questions and expected answers.
2. Run the chatbot or RAG pipeline on those questions.
3. Measure performance using evaluation metrics such as answer relevance, answer correctness, and retrieval quality.

## Workflow Overview

A typical RAG evaluation workflow has three stages:

1. Build a dataset containing user questions and reference answers.
2. Execute the RAG application on that dataset.
3. Evaluate the generated answers and the retrieved context.

## Project Notes

- LLM provider: `Groq`
- Model: `llama-3.1-8b-instant`
- Evaluation and tracing: `LangSmith`
- Knowledge source: Lilian Weng's blog content

We will first validate the workflow inside this notebook and then convert it into a more production-style chatbot architecture.

In [3]:
# Chatbot evaluation system setup
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LLM_MODEL = "llama-3.1-8b-instant"

if not LANGSMITH_API_KEY:
    raise ValueError("LANGSMITH_API_KEY is missing in backend/.env")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing in backend/.env")

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"

print(f"Tracing enabled: {os.environ['LANGSMITH_TRACING']}")
print(f"Groq model: {LLM_MODEL}")

Tracing enabled: true
Groq model: llama-3.1-8b-instant


In [6]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
existing_datasets = list(client.list_datasets(dataset_name=dataset_name))
dataset = existing_datasets[0] if existing_datasets else client.create_dataset(dataset_name)

examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications."},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for tracing, observing, and evaluating LLM applications."},
    },
    {
        "inputs": {"question": "What is Groq used for in this notebook?"},
        "outputs": {"answer": "Groq is the LLM provider used to run the chatbot with the llama-3.1-8b-instant model."},
    },
    {
        "inputs": {"question": "Which model are we using for the chatbot?"},
        "outputs": {"answer": "The chatbot uses the llama-3.1-8b-instant model."},
    },
    {
        "inputs": {"question": "Why do we evaluate a RAG system?"},
        "outputs": {"answer": "We evaluate a RAG system to measure answer quality, grounding, and retrieval performance."},
    },
]

existing_questions = {
    example.inputs.get("question")
    for example in client.list_examples(dataset_id=dataset.id)
}
new_examples = [
    example for example in examples if example["inputs"]["question"] not in existing_questions
]

client.create_examples(
    dataset_id=dataset.id,
    examples=new_examples,
) if new_examples else None

print(f"Dataset ready: {dataset.name}")
print(f"Dataset id: {dataset.id}")
print(f"Examples added in this run: {len(new_examples)}")

Dataset ready: Chatbots Evaluation
Dataset id: f7cce5ac-4280-472b-a1df-96c4581093c2
Examples added in this run: 0


## Define Metrics: LLM as a Judge

In this section, we use a Groq-hosted LLM as the evaluator. The judge model reads the original question, the reference answer, and the chatbot response, then decides whether the response is correct.

We also define a simple `concision` metric to check whether the generated answer stays reasonably short.

In [7]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

PRIMARY_MODEL = "llama-3.1-8b-instant"
COMPARISON_MODEL = "llama-3.3-70b-versatile"
JUDGE_MODEL = PRIMARY_MODEL

eval_instructions = "You are an expert professor specialized in grading students' answers to questions. Reply with only CORRECT or INCORRECT."


def judge_answer(question: str, reference_answer: str, predicted_answer: str, model: str = JUDGE_MODEL) -> str:
    user_content = f"""You are grading the following question:
{question}

Here is the reference answer:
{reference_answer}

Here is the predicted answer:
{predicted_answer}

Respond with only one word: CORRECT or INCORRECT.
Grade:"""

    response = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    )
    return response.choices[0].message.content.strip().upper()


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    grade = judge_answer(
        question=inputs["question"],
        reference_answer=reference_outputs["answer"],
        predicted_answer=outputs["response"],
    )
    score = 1 if grade == "CORRECT" else 0
    return {
        "key": "correctness",
        "score": score,
        "comment": f"Judge result: {grade}",
    }


def concision(outputs: dict, reference_outputs: dict) -> dict:
    generated = outputs["response"].strip()
    expected = reference_outputs["answer"].strip()
    score = int(len(generated) < 2 * len(expected))
    return {
        "key": "concision",
        "score": score,
        "comment": f"generated_length={len(generated)}, reference_length={len(expected)}",
    }


print(f"Judge model: {JUDGE_MODEL}")
print(f"Comparison model: {COMPARISON_MODEL}")

Judge model: llama-3.1-8b-instant
Comparison model: llama-3.3-70b-versatile


## Run Evaluations

Now we define the chatbot application, call it for every dataset example, and run LangSmith evaluations for two Groq models. This gives us side-by-side experiment traces for `correctness` and `concision`.

In [8]:
default_instructions = "Respond to the user's question in a short, concise manner using one short sentence."


def my_app(question: str, model: str = PRIMARY_MODEL, instructions: str = default_instructions) -> str:
    response = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content.strip()


def build_target(model_name: str):
    def ls_target(inputs: dict) -> dict:
        return {"response": my_app(inputs["question"], model=model_name)}

    ls_target.__name__ = f"chatbot_target_{model_name.replace('-', '_').replace('.', '_')}"
    return ls_target

In [9]:
primary_target = build_target(PRIMARY_MODEL)

primary_experiment_results = client.evaluate(
    primary_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="groq-llama-3.1-8b-instant-chatbot",
)

primary_experiment_results

View the evaluation results for experiment: 'groq-llama-3.1-8b-instant-chatbot-8a6e322a' at:
https://smith.langchain.com/o/115aa241-1787-456a-8dee-46be34495f7c/datasets/f7cce5ac-4280-472b-a1df-96c4581093c2/compare?selectedSessions=99ffcb15-4ed1-4d43-ada6-152e029227e3




0it [00:00, ?it/s]

<ExperimentResults groq-llama-3.1-8b-instant-chatbot-8a6e322a>

In [10]:
comparison_target = build_target(COMPARISON_MODEL)

comparison_experiment_results = client.evaluate(
    comparison_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="groq-llama-3.3-70b-versatile-chatbot",
)

comparison_experiment_results

View the evaluation results for experiment: 'groq-llama-3.3-70b-versatile-chatbot-ba7502f3' at:
https://smith.langchain.com/o/115aa241-1787-456a-8dee-46be34495f7c/datasets/f7cce5ac-4280-472b-a1df-96c4581093c2/compare?selectedSessions=2cdde99d-a570-4ff7-baab-ff167fb7f345




0it [00:00, ?it/s]

<ExperimentResults groq-llama-3.3-70b-versatile-chatbot-ba7502f3>

## RAG Evaluation Workflow

For future reference, the RAG evaluation process in this notebook follows a simple three-part structure:

1. Create a test dataset with questions and reference answers.
2. Run the RAG application on that dataset.
3. Measure RAG performance using evaluation metrics.

### Core RAG Pipeline

A RAG system has three main components:

1. Data ingestion: load documents, split them into chunks, and build an index.
2. Retrieval: fetch the most relevant document chunks for a user question.
3. Generation: use the retrieved context to generate an answer.

### Evaluation Structure

- Test data: `question <-> reference answer`
- RAG output: generated answer plus retrieved documents
- Evaluation metrics: LLM-as-a-judge and rule-based checks

In practice, we evaluate the RAG system from multiple angles: correctness against the reference answer, relevance of the answer to the question, relevance of retrieved documents, and whether the final answer is grounded in the retrieved context.

## Build the RAG Pipeline

In this section, we create a simple RAG pipeline using Lilian Weng's blog posts as the source documents. We use:

- `WebBaseLoader` to load web pages
- `RecursiveCharacterTextSplitter` to create chunks
- `HuggingFaceEmbeddings` for local embeddings
- `InMemoryVectorStore` as the vector index
- `ChatGroq` with `llama-3.1-8b-instant` for answer generation

This keeps the pipeline compatible with your Groq-based setup and avoids any dependency on OpenAI embeddings or OpenAI chat models.

In [12]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

rag_urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

loaded_docs = [WebBaseLoader(url).load() for url in rag_urls]
docs_list = [doc for doc_group in loaded_docs for doc in doc_group]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=0,
)
doc_splits = text_splitter.split_documents(docs_list)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

rag_llm = ChatGroq(model=PRIMARY_MODEL, temperature=0)

print(f"Source documents loaded: {len(docs_list)}")
print(f"Document chunks created: {len(doc_splits)}")
print(f"RAG generation model: {PRIMARY_MODEL}")

C:\Users\Gowtham\AppData\Local\Temp\ipykernel_8276\4189603017.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Source documents loaded: 3
Document chunks created: 730
RAG generation model: llama-3.1-8b-instant


In [13]:
retriever.invoke("what are agents")

[Document(id='fa23b3fc-71b4-433d-aa79-4b499aaf0370', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

## Define the RAG Bot

The `rag_bot` function retrieves relevant documents for a question, combines them into a prompt, and asks the Groq model to answer using only that retrieved context. We return both the generated answer and the retrieved documents so they can be evaluated later in LangSmith.

In [14]:
from langsmith import traceable


@traceable()
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    docs_string = "\n\n".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
Use the following source documents to answer the user's question.
If you do not know the answer, say that you do not know.
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    ai_msg = rag_llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])

    return {
        "answer": ai_msg.content,
        "documents": docs,
    }


In [15]:
rag_bot("What are agents?")

{'answer': 'I do not know.',
 'documents': [Document(id='11342c1f-d204-46b3-b6ea-e54540f76d2e', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from 

## Create the RAG Evaluation Dataset

This dataset is separate from the earlier chatbot-only dataset. It stores RAG-focused evaluation examples, where each example contains a question and a reference answer. We will use this dataset in the next step when we evaluate the RAG pipeline with dedicated metrics.

In [16]:
rag_dataset_name = "RAG Test Evaluation"

rag_examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection?"},
        "outputs": {"answer": "ReAct integrates reasoning and acting, taking actions such as using tools and then reasoning over the observed results."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The main few-shot prompting biases are majority label bias, recency bias, and common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five adversarial attack types are token manipulation, gradient-based attacks, jailbreak prompting, human red-teaming, and model red-teaming."},
    },
]

existing_rag_datasets = list(client.list_datasets(dataset_name=rag_dataset_name))
rag_dataset = existing_rag_datasets[0] if existing_rag_datasets else client.create_dataset(dataset_name=rag_dataset_name)

existing_rag_questions = {
    example.inputs.get("question")
    for example in client.list_examples(dataset_id=rag_dataset.id)
}
new_rag_examples = [
    example for example in rag_examples if example["inputs"]["question"] not in existing_rag_questions
]

client.create_examples(
    dataset_id=rag_dataset.id,
    examples=new_rag_examples,
) if new_rag_examples else None

print(f"RAG dataset ready: {rag_dataset.name}")
print(f"RAG dataset id: {rag_dataset.id}")
print(f"Examples added in this run: {len(new_rag_examples)}")

RAG dataset ready: RAG Test Evaluation
RAG dataset id: 1e1c0824-6c6c-4ca1-8d8a-9e89e2b30019
Examples added in this run: 3


## RAG Evaluators and Metrics

In this section, we define the four main evaluators used for RAG assessment in LangSmith.

1. `correctness`: compares the generated answer with the reference answer.
2. `relevance`: checks whether the generated answer addresses the input question.
3. `groundedness`: checks whether the generated answer is supported by the retrieved documents.
4. `retrieval_relevance`: checks whether the retrieved documents are relevant to the input question.

All evaluators below use a Groq-backed LLM judge so the notebook stays aligned with your current setup.

### Correctness: Response vs Reference Answer

Goal: measure how correct the RAG answer is relative to the ground-truth answer in the dataset.

- Requires a reference answer
- Uses LLM-as-a-judge to assess factual correctness

In [28]:
import json
import re

from typing_extensions import Annotated, TypedDict


class CorrectnessGrade(TypedDict):
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]


correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH ANSWER, and the STUDENT ANSWER.

Here is the grading criteria to follow:
(1) Grade the student answer based only on its factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain conflicting statements.
(3) It is acceptable if the student answer contains more information than the ground truth answer, as long as it remains factually accurate.

Correctness:
A correctness value of True means that the student answer meets all of the criteria.
A correctness value of False means that the student answer does not meet all of the criteria.
"""

JSON_GRADING_INSTRUCTIONS = """Return valid JSON only.
Do not wrap the JSON in markdown.
Do not call tools.
Do not add any text before or after the JSON.
Use lowercase true or false for booleans.
Keep the JSON to a single line.
Return only the requested boolean field.
Example: {\"correct\": true}
"""


def _extract_json_object(text: str) -> dict:
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise

        candidate = match.group(0)
        candidate = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", " ", candidate)
        return json.loads(candidate)


def _judge_with_json(instructions: str, payload: str, result_key: str) -> dict:
    response = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "system", "content": JSON_GRADING_INSTRUCTIONS},
            {"role": "user", "content": payload},
        ],
    )
    content = response.choices[0].message.content
    parsed = _extract_json_object(content)
    return {result_key: bool(parsed[result_key])}


def _serialize_documents(documents: list, max_chars: int = 6000) -> str:
    chunks = []
    total = 0
    for index, doc in enumerate(documents, start=1):
        piece = f"[Document {index}]\n{doc.page_content.strip()}"
        if total + len(piece) > max_chars:
            remaining = max_chars - total
            if remaining > 0:
                chunks.append(piece[:remaining])
            break
        chunks.append(piece)
        total += len(piece)
    return "\n\n".join(chunks)


def rag_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    answers = f"""QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    grade = _judge_with_json(
        correctness_instructions + "\n\nReturn a JSON object with one key only: correct.",
        answers,
        "correct",
    )
    return grade["correct"]


### Relevance: Response vs Input

Goal: measure whether the generated answer actually addresses the user's question.

- Does not require a reference answer
- Measures helpfulness and alignment to the question

In [29]:
class RelevanceGrade(TypedDict):
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]


relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grading criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION.
(2) Ensure the STUDENT ANSWER helps answer the QUESTION.

Relevance:
A relevance value of True means that the student answer meets all of the criteria.
A relevance value of False means that the student answer does not meet all of the criteria.
"""

def rag_relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = _judge_with_json(
        relevance_instructions + "\n\nReturn a JSON object with one key only: relevant.",
        answer,
        "relevant",
    )
    return grade["relevant"]


### Groundedness: Response vs Retrieved Documents

Goal: measure whether the answer is supported by the retrieved context and avoids hallucinations.

- Does not require a reference answer
- Uses retrieved documents as the factual source

In [30]:
class GroundedGrade(TypedDict):
    grounded: Annotated[bool, ..., "Provide the score on whether the answer hallucinates from the documents"]


grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grading criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain hallucinated information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student answer meets all of the criteria.
A grounded value of False means that the student answer does not meet all of the criteria.
"""

def rag_groundedness(inputs: dict, outputs: dict) -> bool:
    doc_string = _serialize_documents(outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = _judge_with_json(
        grounded_instructions + "\n\nReturn a JSON object with one key only: grounded.",
        answer,
        "grounded",
    )
    return grade["grounded"]


### Retrieval Relevance: Retrieved Documents vs Input

Goal: measure whether the documents returned by the retriever are relevant to the user's question.

- Does not require a reference answer
- Focuses on retrieval quality before generation

In [31]:
class RetrievalRelevanceGrade(TypedDict):
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]


retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grading criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION.
(2) If the facts contain any keywords or semantic meaning related to the QUESTION, consider them relevant.
(3) It is acceptable if the facts contain some unrelated information as long as condition (2) is met.

Relevance:
A relevance value of True means that the FACTS contain any keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.
"""

def rag_retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    doc_string = _serialize_documents(outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    grade = _judge_with_json(
        retrieval_relevance_instructions + "\n\nReturn a JSON object with one key only: relevant.",
        answer,
        "relevant",
    )
    return grade["relevant"]


## Run the RAG Evaluation

Now we evaluate the full RAG pipeline on the `RAG Test Evaluation` dataset. Each question is passed into `rag_bot`, and LangSmith records the generated answer, retrieved documents, and the evaluator scores.

This gives you a full experiment view across:

- correctness
- relevance
- groundedness
- retrieval relevance

In [32]:
def rag_target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])


rag_experiment_results = client.evaluate(
    rag_target,
    data=rag_dataset_name,
    evaluators=[rag_correctness, rag_groundedness, rag_relevance, rag_retrieval_relevance],
    experiment_prefix="groq-rag-evaluation-json-judge",
    metadata={"provider": "groq", "model": PRIMARY_MODEL, "pipeline": "rag"},
)

rag_experiment_results

View the evaluation results for experiment: 'groq-rag-evaluation-json-judge-e25fdb72' at:
https://smith.langchain.com/o/115aa241-1787-456a-8dee-46be34495f7c/datasets/1e1c0824-6c6c-4ca1-8d8a-9e89e2b30019/compare?selectedSessions=e0fa9d27-ed1c-4377-8c21-f32d47a38278




0it [00:00, ?it/s]

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.rag_correctness,feedback.rag_groundedness,feedback.rag_relevance,feedback.rag_retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,"According to Zhao et al. (2021), several biase...",[page_content='Zhao et al. (2021) investigated...,None,The main few-shot prompting biases are majorit...,True,True,True,True,1.980729,1889becb-652c-4d31-ab64-49c30b5e7133,019d9554-2d74-7ca1-851f-0bcc8d0f5efc
1,What are five types of adversarial attacks?,I do not know the five types of adversarial at...,[page_content='The most common way to mitigate...,None,Five adversarial attack types are token manipu...,False,False,False,False,0.619550,83e61096-f87f-457a-a0a6-e58c9d0f6db2,019d9554-3ac3-73e3-8ae7-a964e4693133
2,How does the ReAct agent use self-reflection?,I do not know. The provided source documents d...,[page_content='Self-reflection is created by s...,None,"ReAct integrates reasoning and acting, taking ...",False,False,False,True,0.617709,b33b3e48-146b-4731-b4a2-fd7a0d547c17,019d9554-41a1-79f1-816a-1020ddcd26a4


In [33]:
rag_experiment_results.to_pandas()

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.rag_correctness,feedback.rag_groundedness,feedback.rag_relevance,feedback.rag_retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,"According to Zhao et al. (2021), several biase...",[page_content='Zhao et al. (2021) investigated...,None,The main few-shot prompting biases are majorit...,True,True,True,True,1.980729,1889becb-652c-4d31-ab64-49c30b5e7133,019d9554-2d74-7ca1-851f-0bcc8d0f5efc
1,What are five types of adversarial attacks?,I do not know the five types of adversarial at...,[page_content='The most common way to mitigate...,None,Five adversarial attack types are token manipu...,False,False,False,False,0.619550,83e61096-f87f-457a-a0a6-e58c9d0f6db2,019d9554-3ac3-73e3-8ae7-a964e4693133
2,How does the ReAct agent use self-reflection?,I do not know. The provided source documents d...,[page_content='Self-reflection is created by s...,None,"ReAct integrates reasoning and acting, taking ...",False,False,False,True,0.617709,b33b3e48-146b-4731-b4a2-fd7a0d547c17,019d9554-41a1-79f1-816a-1020ddcd26a4
